## The notebook Section-by-section

**0-1**: Setup. Installs packages, defines the 6 models, the 3 grid conditions (metrics, likes_only, likes_only_noise), and the 7 engagement scales.

**2**: Load the data. Reads all your raw JSON result files into one big table — one row per trial, across every model/condition/image/scale-pair. Also computes disparity: a single number capturing "how much more engagement did the incorrect post have," on a log scale (so 10-vs-100 and 10,000-vs-100,000 count as the same disparity, just different absolute sizes).

**3**: Sanity check. Recomputes the diagonal/collapse numbers you already know from THESIS.md by hand, using the freshly-loaded data — purely to confirm nothing broke while loading. Not a new finding, just a "did I load this correctly" check.

**4-4a**: The main model. This is the core of the notebook. It asks: does the size of the engagement gap predict whether the model picks the correct post, and does that relationship differ by model, and by signal type (metrics vs. likes-only vs. likes-only-with-noise)? The coefficient table turns the raw statistical output into odds ratios — "for every 10x increase in the engagement gap, the odds of choosing correctly change by X%" — which is a much stronger claim than a heatmap of percentages.

**5**: Robustness check. Re-runs the same model but groups the data differently (by image alone, instead of image+model). If the results look similar either way, you can trust the finding isn't an artifact of how you chose to group things.

**6**: The two headline hypothesis tests. Formal tests for the two things your thesis is actually trying to prove:
- Does the disparity effect differ significantly across models (i.e., is scale-dependence within a model family, like Gemma, a real statistical effect and not just eyeballing a chart)?
- Does the disparity effect differ significantly by signal type (i.e., is the metrics-vs-likes-only asymmetry you found real)?

**7**: Baseline condition. A separate, simpler model just for the "no engagement shown at all" condition, since there's no disparity to speak of there.

**8**: Export. Saves everything to CSV files for pasting into the thesis.

**9/9a** Which combinations show high magnitude collapse pattern.

**10** (just recovered): The corner-sum test — this is the one that most directly answers "is the model just following popularity, or does correctness matter independently?" It compares a cell to its mirror (swap which post has more engagement) — if they always sum to ~100%, the model is purelyengagement-driven; if the sum is meaningfully above 100%, correctness is pulling extra weight.

**11** (just recovered): Simplest of the three — just checks whether the model beats a coin flip (50%) when both posts have identical engagement.

# GEE Analysis — Engagement-Disparity Conformity Effect Across Models

Companion notebook to `docs/THESIS.md` (Chapter 8, item 3) and `docs/STATS_ANALYSIS_NOTES.md`.

**Purpose:** replace the descriptive percentages/heatmaps and per-model sign tests already in the thesis with a single formal statistical model — a binomial GEE (Generalized Estimating Equations) clustered by image — that can test the thesis's central hypotheses directly, with proper handling of the fact that the same 100 images are reused across every grid cell, signal type, and model (see `docs/THESIS.md` Appendix C.1 for a plain-language explanation of why that reuse is a problem, and `docs/STATS_ANALYSIS_NOTES.md` for the method comparison that led to choosing GEE as the starting point).

**Why GEE specifically (not GLMM):** GEE was chosen to start because it converges even when some model/condition combinations produce a constant 0% or 100% answer (several models do, under some phrasings — Section 6.3), which breaks standard maximum-likelihood GLMM fitting via complete separation. GLMM remains a possible follow-up once/if the degenerate cells are handled separately (e.g. dropped, or fit with Firth-penalized regression) — see the pros/cons table in `docs/STATS_ANALYSIS_NOTES.md`.

**Where this runs:** the local dev sandbox used for the rest of this project's Claude Code session has no `pip`/`numpy`/`pandas`/`statsmodels` available and no way to install them. This notebook is meant to be run on your GPU server (or any machine where you can `pip install`), not in that sandbox.

## 0. Setup

Run the install cell once per environment. If `numpy`/`pandas`/`scipy`/`statsmodels` are already available on your server, skip it.

In [1]:
# One-time setup — skip if already installed
!pip install --quiet numpy pandas scipy statsmodels


/bin/bash: line 1: pip: command not found


In [2]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)


## 1. Config — models and paths

Adjust `REPO_ROOT` if this notebook is moved or run from a different checkout.

In [3]:
REPO_ROOT = Path("..").resolve()  
assert (REPO_ROOT / "experiments" / "e1").exists(), f"Unexpected REPO_ROOT: {REPO_ROOT}"

MODELS = [
    "gemma4-12b",
    "gemma4-e4b",
    "qwen3-vl-4b",
    "qwen3-vl-8b",
    "ministral-3-8b",
    "ministral-3-14b",
]

# Signal-type conditions that vary engagement scale over a 7x7 grid (correct_scale x incorrect_scale).
# 'baseline' is handled separately below — it shows no engagement numbers at all, so there is no
# disparity to speak of; treating it as "disparity = 0" would silently conflate "no numbers shown"
# with "equal numbers shown", which are different conditions. Revisit this if you disagree.
GRID_CONDITIONS = ["metrics","likes_only_noise"] # excluded "likes_only"

SCALES = [0, 10, 100, 1000, 10000, 100000, 1000000]


## 2. Load the disparity-grid data (metrics / likes_only / likes_only_noise)

Builds one long-format row per trial: which model, which image, the engagement disparity (continuous), which signal type, and whether the model chose the correct post.

**Disparity definition** (matches `docs/STATS_ANALYSIS_NOTES.md`): `disparity = log10(incorrect_scale + 1) - log10(correct_scale + 1)`. Positive disparity means the *incorrect* post has the engagement advantage — this is the direction where conformity bias, if present, should show up as a *lower* probability of choosing correct. Zero means equal engagement (the diagonal of the grid).

In [4]:
def load_grid_condition(model_dir: str, condition: str) -> pd.DataFrame:
    path = REPO_ROOT / "experiments" / "e1" / model_dir / "outputs" / f"e1_results_{condition}_paired.json"
    records = json.loads(path.read_text())
    rows = []
    for r in records:
        cs, ics = r["correct_scale"], r["incorrect_scale"]
        disparity = math.log10(ics + 1) - math.log10(cs + 1)
        rows.append({
            "model": model_dir,
            "image_num": r["num"],
            "signal_type": condition,
            "correct_scale": cs,
            "incorrect_scale": ics,
            "disparity": disparity,
            "chose_correct": 1 if r["liked_variant"] == "correct" else 0,
        })
    return pd.DataFrame(rows)


grid_frames = []
for model_dir in MODELS:
    for condition in GRID_CONDITIONS:
        grid_frames.append(load_grid_condition(model_dir, condition))

df = pd.concat(grid_frames, ignore_index=True)
# Cluster unit: same image, reused across all 49 cells x 3 signal types, WITHIN a given model.
# This is the more conservative reading of the pseudo-replication concern in STATS_ANALYSIS_NOTES.md —
# it protects against "this model has an idiosyncratic quirk on this image" being counted many times.
# An alternative (clustering by image_num alone, shared across models) is tried as a robustness check
# in section 5 below — the two should be compared before trusting either one exclusively.
df["cluster"] = df["model"] + "__" + df["image_num"]

print(df.shape)
df


(58800, 8)


,model,image_num,signal_type,correct_scale,incorrect_scale,disparity,chose_correct,cluster
0,gemma4-12b,001,metrics,0,0,0.0,1,gemma4-12b__001
1,gemma4-12b,003,metrics,0,0,0.0,0,gemma4-12b__003
2,gemma4-12b,004,metrics,0,0,0.0,1,gemma4-12b__004
3,gemma4-12b,005,metrics,0,0,0.0,1,gemma4-12b__005
4,gemma4-12b,006,metrics,0,0,0.0,1,gemma4-12b__006
...,...,...,...,...,...,...,...,...
58795,ministral-3-14b,096,likes_only_noise,1000000,1000000,0.0,1,ministral-3-14b__096
58796,ministral-3-14b,097,likes_only_noise,1000000,1000000,0.0,0,ministral-3-14b__097
58797,ministral-3-14b,098,likes_only_noise,1000000,1000000,0.0,0,ministral-3-14b__098
58798,ministral-3-14b,099,likes_only_noise,1000000,1000000,0.0,1,ministral-3-14b__099


## 3. Sanity check against numbers already in the thesis

Before trusting any model output, reproduce a few descriptive numbers already reported in `docs/THESIS.md` §6.1 (the six-model diagonal/above-diagonal/collapse table) directly from this dataframe. If these don't match, something is wrong with the loading code above — stop and debug before fitting anything.

In [5]:
def diagonal_above_collapse(model_dir: str, condition: str = "metrics"):
    sub = df[(df["model"] == model_dir) & (df["signal_type"] == condition)]
    diag = sub[sub["disparity"] == 0]["chose_correct"].mean() * 100
    above = sub[sub["disparity"] > 0]["chose_correct"].mean() * 100
    return diag, above, diag - above


print(f"{'model':24s} {'diagonal':>10s} {'above-diag':>12s} {'collapse':>10s}")
for m in MODELS:
    d, a, c = diagonal_above_collapse(m)
    print(f"{m:24s} {d:9.1f}% {a:11.1f}% {c:9.1f} pt")

# Expected from THESIS.md Section 6.1 (already verified against raw JSON on 2026-07-15):
#   Gemma-12B    82.7% / 84.7% / -2.0pt (no collapse)
#   Gemma-E4B    52.3% / 1.0%  / 51.2pt
#   Qwen3-VL-4B  45.0% / 9.6%  / 35.4pt
#   Qwen3-VL-8B  55.0% / 1.4%  / 53.6pt
#   Pixtral-12B  51.7% / 22.6% / 29.1pt
#   Mistral-24B  51.7% / 7.5%  / 44.2pt


model                      diagonal   above-diag   collapse
gemma4-12b                    82.7%        84.7%      -2.0 pt
gemma4-e4b                    52.3%         1.0%      51.2 pt
qwen3-vl-4b                   45.0%         9.6%      35.4 pt
qwen3-vl-8b                   55.0%         1.4%      53.6 pt
ministral-3-8b                51.7%         4.5%      47.2 pt
ministral-3-14b               51.7%         0.0%      51.7 pt


## 4-4a: The main model. 

This is the core of the notebook. It asks: does the size of the engagement gap predict whether the model picks the correct post, and does that relationship differ by model, and by signal type (metrics vs. likes-only vs. likes-only-with-noise)? The coefficient table turns the raw statistical output into odds ratios — "for every 10x increase in the engagement gap, the odds of choosing correctly change by X%" — which is a much stronger claim than a heatmap of percentages.


## 4. Primary GEE model

Formula: `chose_correct ~ C(model) * disparity + C(signal_type) * disparity`

- `C(model) * disparity` tests the Section 6.1 scale/family-dependence question directly: does the slope of the disparity effect differ significantly by model, rather than relying on comparing six heatmaps by eye?
- `C(signal_type) * disparity` tests the Section 6.2 metrics-vs-likes-only asymmetry the same way.
- `groups=df["cluster"]` (model+image) and `cov_struct=Independence()` implement the clustering fix described in Appendix C.1/C.2 of the thesis — treats all trials from the same model on the same image as correlated, and computes robust ("sandwich") standard errors accordingly, instead of pretending every row is independent.

This will likely take a little while to fit given ~88,000 rows (6 models x 3 conditions x 4,900 trials each).

In [6]:
gee_model = smf.gee(
    "chose_correct ~ C(model, Treatment(reference='gemma4-12b')) * disparity + C(signal_type, Treatment(reference='metrics')) * disparity",
    groups="cluster",
    data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Independence(),
)
gee_result = gee_model.fit()
print(gee_result.summary())


                               GEE Regression Results                              
Dep. Variable:               chose_correct   No. Observations:                58800
Model:                                 GEE   No. clusters:                      600
Method:                        Generalized   Min. cluster size:                  98
                      Estimating Equations   Max. cluster size:                  98
Family:                           Binomial   Mean cluster size:                98.0
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Wed, 19 Aug 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         16:31:18
                                                                                   coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

### 4a. Clean coefficient table (odds ratios + 95% CI)

Raw logistic-regression coefficients are in log-odds units, which are not intuitive. This converts each coefficient to an odds ratio (`exp(coef)`) with a 95% confidence interval — e.g. an odds ratio of 0.5 for `disparity` means each one-unit increase in disparity (one order of magnitude of engagement favoring the incorrect post) roughly halves the odds of choosing the correct post, holding everything else constant.

In [7]:
def coef_table(result):
    conf = result.conf_int()
    out = pd.DataFrame({
        "coef": result.params,
        "std_err": result.bse,
        "p_value": result.pvalues,
        "OR": np.exp(result.params),
        "OR_2.5%": np.exp(conf[0]),
        "OR_97.5%": np.exp(conf[1]),
    })
    return out.round(4)


coef_table(gee_result)


,coef,std_err,p_value,OR,OR_2.5%,OR_97.5%
Intercept,1.9478,0.1145,0.0000,7.0131,5.6034,8.7775
"C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]",-1.9226,0.1200,0.0000,0.1462,0.1156,0.1850
"C(model, Treatment(reference='gemma4-12b'))[T.ministral-3-14b]",-1.9827,0.1238,0.0000,0.1377,0.1080,0.1755
"C(model, Treatment(reference='gemma4-12b'))[T.ministral-3-8b]",-2.0542,0.1208,0.0000,0.1282,0.1012,0.1624
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-4b]",-1.9558,0.1207,0.0000,0.1415,0.1117,0.1792
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-8b]",-1.9271,0.1216,0.0000,0.1456,0.1147,0.1848
"C(signal_type, Treatment(reference='metrics'))[T.likes_only_noise]",-0.0934,0.0253,0.0002,0.9108,0.8667,0.9571
disparity,-0.3276,0.0307,0.0000,0.7206,0.6785,0.7653
"C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]:disparity",-1.1087,0.0500,0.0000,0.3300,0.2992,0.3639
"C(model, Treatment(reference='gemma4-12b'))[T.ministral-3-14b]:disparity",-1.6776,0.0747,0.0000,0.1868,0.1614,0.2163


## 5. Robustness check — alternate clustering (image only, shared across models)

`docs/STATS_ANALYSIS_NOTES.md` motivates clustering by image because the same 100 images repeat across every cell/condition. Section 4 above clusters by *(model, image)*, on the reasoning that a given model's own idiosyncratic reaction to a given image is the main repeated-measures concern. An arguably more conservative alternative is to cluster by *image alone*, treating any image-level quirk as potentially shared across models too (e.g. a chart that's just visually ambiguous for every model, not only one).

Run both and compare — if the standard errors and significance conclusions are similar either way, the choice doesn't matter much for the substantive conclusions. If they diverge meaningfully, that's worth flagging explicitly rather than picking whichever one supports the preferred story.

In [8]:
gee_model_altcluster = smf.gee(
    "chose_correct ~ C(model, Treatment(reference='gemma4-12b')) * disparity + C(signal_type, Treatment(reference='metrics')) * disparity",
    groups="image_num",  # clustering by image only, pooled across all 6 models
    data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Independence(),
)
gee_result_altcluster = gee_model_altcluster.fit()

comparison = pd.DataFrame({
    "coef": gee_result.params,
    "SE (model+image cluster)": gee_result.bse,
    "SE (image-only cluster)": gee_result_altcluster.bse,
})
comparison["SE ratio"] = comparison["SE (image-only cluster)"] / comparison["SE (model+image cluster)"]
comparison.round(4)


,coef,SE (model+image cluster),SE (image-only cluster),SE ratio
Intercept,1.9478,0.1145,0.1120,0.9783
"C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]",-1.9226,0.1200,0.1159,0.9660
"C(model, Treatment(reference='gemma4-12b'))[T.ministral-3-14b]",-1.9827,0.1238,0.1223,0.9876
"C(model, Treatment(reference='gemma4-12b'))[T.ministral-3-8b]",-2.0542,0.1208,0.1331,1.1016
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-4b]",-1.9558,0.1207,0.1133,0.9389
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-8b]",-1.9271,0.1216,0.1081,0.8888
"C(signal_type, Treatment(reference='metrics'))[T.likes_only_noise]",-0.0934,0.0253,0.0224,0.8835
disparity,-0.3276,0.0307,0.0291,0.9473
"C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]:disparity",-1.1087,0.0500,0.0494,0.9876
"C(model, Treatment(reference='gemma4-12b'))[T.ministral-3-14b]:disparity",-1.6776,0.0747,0.0730,0.9771


## 6. Formal tests of the thesis's two central hypotheses

Rather than reading individual interaction-term p-values above one at a time, test each hypothesis as a *joint* Wald test across all of that hypothesis's interaction terms together — this is the direct, formal version of what Section 6.1's heatmap-by-heatmap comparison and Section 6.2's metrics-vs-likes-only description currently do by eye/prose.

In [9]:
# Joint test: does the disparity slope differ significantly across models at all?
# (i.e. is there any real model x disparity interaction, not just "models look different on a heatmap")
model_disparity_terms = [name for name in gee_result.params.index if "C(model" in name and "disparity" in name and ":" in name]
print("Terms included in the model x disparity joint test:")
for t in model_disparity_terms:
    print(" -", t)

wald_model_disparity = gee_result.wald_test(model_disparity_terms, scalar=True)
print("\nJoint Wald test (model x disparity):", wald_model_disparity)


Terms included in the model x disparity joint test:
 - C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]:disparity
 - C(model, Treatment(reference='gemma4-12b'))[T.ministral-3-14b]:disparity
 - C(model, Treatment(reference='gemma4-12b'))[T.ministral-3-8b]:disparity
 - C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-4b]:disparity
 - C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-8b]:disparity

Joint Wald test (model x disparity): <Wald test (chi2): statistic=1379.2575286227454, p-value=4.2979784452644047e-296, df_denom=5>


In [10]:
# Joint test: does the disparity slope differ significantly by signal type?
# (the Section 6.2 metrics-vs-likes-only-vs-noise asymmetry, tested formally instead of by eye)
signal_disparity_terms = [name for name in gee_result.params.index if "C(signal_type" in name and ":disparity" in name]
print("Terms included in the signal_type x disparity joint test:")
for t in signal_disparity_terms:
    print(" -", t)

wald_signal_disparity = gee_result.wald_test(signal_disparity_terms, scalar=True)
print("\nJoint Wald test (signal_type x disparity):", wald_signal_disparity)


Terms included in the signal_type x disparity joint test:
 - C(signal_type, Treatment(reference='metrics'))[T.likes_only_noise]:disparity

Joint Wald test (signal_type x disparity): <Wald test (chi2): statistic=154.87668115958292, p-value=1.489850864509115e-35, df_denom=1>


## 7. Baseline condition (no engagement shown at all) — separate, simpler model

The `baseline` condition shows no engagement numbers at all, so it has no `disparity` value — it doesn't belong in the grid model above. This is the same comparison already covered descriptively in `docs/THESIS.md` (each model's unscaled A/B baseline, e.g. Gemma-12B 83%/17%), reproduced here as a single clustered logistic model across all six models for a formal cross-model comparison with a p-value, rather than six numbers compared by eye. There is only one trial per image per model in this condition (no repeated cells), so clustering only matters if you consider the same image across the 6 models as correlated — included for consistency, but the correction will be minor here.

In [11]:
def load_baseline(model_dir: str) -> pd.DataFrame:
    path = REPO_ROOT / "experiments" / "e1" / model_dir / "outputs" / "e1_results_baseline_paired.json"
    records = json.loads(path.read_text())
    rows = [{
        "model": model_dir,
        "image_num": r["num"],
        "chose_correct": 1 if r["liked_variant"] == "correct" else 0,
    } for r in records]
    return pd.DataFrame(rows)


baseline_df = pd.concat([load_baseline(m) for m in MODELS], ignore_index=True)

baseline_gee = smf.gee(
    "chose_correct ~ C(model, Treatment(reference='gemma4-12b'))",
    groups="image_num",
    data=baseline_df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Independence(),
)
baseline_result = baseline_gee.fit()
coef_table(baseline_result)


,coef,std_err,p_value,OR,OR_2.5%,OR_97.5%
Intercept,1.5856,0.2662,0.0000,4.8824,2.8975,8.2269
"C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]",-1.1383,0.3748,0.0024,0.3204,0.1537,0.6679
"C(model, Treatment(reference='gemma4-12b'))[T.ministral-3-14b]",-1.3445,0.3781,0.0004,0.2607,0.1242,0.5470
"C(model, Treatment(reference='gemma4-12b'))[T.ministral-3-8b]",-1.3445,0.3781,0.0004,0.2607,0.1242,0.5470
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-4b]",-2.2047,0.3204,0.0000,0.1103,0.0589,0.2067
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-8b]",-1.4655,0.2872,0.0000,0.2310,0.1315,0.4056


## 8. Export results

Saves the primary model's coefficient table and both Wald test results to `statistical_analysis/outputs/` for pasting into `docs/THESIS.md` / `docs/STATS_ANALYSIS_NOTES.md` once reviewed. Nothing here is written back into the thesis automatically — treat this as a draft result to check against the existing descriptive findings (Section 3 above) before it replaces or supplements anything in Chapter 6.

In [12]:
out_dir = REPO_ROOT / "statistical_analysis" / "outputs"
out_dir.mkdir(exist_ok=True)

coef_table(gee_result).to_csv(out_dir / "gee_primary_model_coefficients.csv")
comparison.to_csv(out_dir / "gee_clustering_robustness_check.csv")
coef_table(baseline_result).to_csv(out_dir / "gee_baseline_model_coefficients.csv")

with open(out_dir / "wald_tests.txt", "w") as f:
    f.write("Joint Wald test (model x disparity):\n")
    f.write(str(wald_model_disparity) + "\n\n")
    f.write("Joint Wald test (signal_type x disparity):\n")
    f.write(str(wald_signal_disparity) + "\n")

print(f"Saved outputs to {out_dir}")


Saved outputs to /home/dbvis-support/ClaudeProjects/conformity-llms-facebook-posts/statistical_analysis/outputs


## 9. Formal test of the recurring high-magnitude discrimination-collapse pattern (Section 6.4)

`docs/THESIS.md` Section 6.4 documents a pattern, found by inspecting grid slices by eye, in which five of the eighteen model x signal-type combinations show accuracy converging toward chance once *both* posts' engagement counts get large (roughly 10,000+), regardless of which post is favored — while the other thirteen combinations stay confidently saturated at every scale. This section tests that pattern formally instead of continuing to eyeball heatmap slices.

**Why `disparity` alone (or `disparity**2`) cannot detect this.** `disparity` is a log-*ratio* (`log10(incorrect+1) - log10(correct+1)`), so it is scale-invariant by construction: the pair "10 vs. 100" and the pair "10,000 vs. 100,000" have the *identical* disparity value, even though the collapse only shows up in the second, much larger pair. A squared-disparity term is still only a function of the ratio, so it cannot pick this up either.

**The fix: add a separate term for the absolute scale of the numbers shown, and test its interaction with disparity.** `avg_scale = (log10(correct_scale + 1) + log10(incorrect_scale + 1)) / 2` captures how large the two numbers are, independent of their ratio (0 at "0 vs. 0", up to 6 at "1,000,000 vs. 1,000,000"). Fitting `chose_correct ~ disparity * avg_scale` **separately for each of the 18 model x signal-type combinations** lets us read off, for each one, whether the `disparity:avg_scale` interaction is significantly negative — i.e. whether the disparity effect genuinely weakens as the numbers get larger. That is the direct, formal test of what Section 6.4 currently only shows via two representative grid slices per flagged combination.

In [13]:
df["avg_scale"] = (
    np.log10(df["correct_scale"] + 1) + np.log10(df["incorrect_scale"] + 1)
) / 2

# Sanity check: avg_scale should range from 0 (0 vs 0) to 6 (1M vs 1M)
print(df["avg_scale"].min(), df["avg_scale"].max())
df[["model", "signal_type", "correct_scale", "incorrect_scale", "disparity", "avg_scale"]].head()


0.0 6.0000004342942646


,model,signal_type,correct_scale,incorrect_scale,disparity,avg_scale
0,gemma4-12b,metrics,0,0,0.0,0.0
1,gemma4-12b,metrics,0,0,0.0,0.0
2,gemma4-12b,metrics,0,0,0.0,0.0
3,gemma4-12b,metrics,0,0,0.0,0.0
4,gemma4-12b,metrics,0,0,0.0,0.0


In [14]:
scale_interaction_rows = []

for model_dir in MODELS:
    for condition in GRID_CONDITIONS:
        sub = df[(df["model"] == model_dir) & (df["signal_type"] == condition)].copy()
        try:
            m = smf.gee(
                "chose_correct ~ disparity * avg_scale",
                groups="image_num",
                data=sub,
                family=sm.families.Binomial(),
                cov_struct=sm.cov_struct.Independence(),
            )
            r = m.fit()
            term = "disparity:avg_scale"
            scale_interaction_rows.append({
                "model": model_dir,
                "signal_type": condition,
                "coef": r.params[term],
                "std_err": r.bse[term],
                "p_value": r.pvalues[term],
                "n": len(sub),
                "converged": True,
            })
        except Exception as e:
            scale_interaction_rows.append({
                "model": model_dir,
                "signal_type": condition,
                "coef": np.nan,
                "std_err": np.nan,
                "p_value": np.nan,
                "n": len(sub),
                "converged": False,
                "error": str(e),
            })
            print(f"FAILED to fit {model_dir} / {condition}: {e}")

scale_interaction = pd.DataFrame(scale_interaction_rows).sort_values("p_value")
scale_interaction


,model,signal_type,coef,std_err,p_value,n,converged
3,gemma4-e4b,likes_only_noise,0.341236,0.019971,1.879604e-65,4900,True
4,qwen3-vl-4b,metrics,0.502836,0.063837,3.358195e-15,4900,True
2,gemma4-e4b,metrics,0.886712,0.141918,4.155712e-10,4900,True
5,qwen3-vl-4b,likes_only_noise,0.137265,0.029819,4.158338e-06,4900,True
0,gemma4-12b,metrics,0.070058,0.018748,1.863091e-04,4900,True
11,ministral-3-14b,likes_only_noise,0.111309,0.035665,1.802726e-03,4900,True
7,qwen3-vl-8b,likes_only_noise,0.041935,0.015752,7.761402e-03,4900,True
1,gemma4-12b,likes_only_noise,0.037831,0.014575,9.444463e-03,4900,True
10,ministral-3-14b,metrics,-0.985766,0.506487,5.162108e-02,4900,True
6,qwen3-vl-8b,metrics,0.080525,0.046474,8.314765e-02,4900,True


### 9a. Which combinations show the high-magnitude collapse pattern

Formal result from the `disparity * avg_scale` model above: a combination shows the pattern if its
`disparity:avg_scale` coefficient is negative **and** significant (p < 0.05) — meaning the disparity
effect measurably weakens as the shown engagement numbers get larger, not just as their ratio changes.
Sorted by p-value so the strongest evidence is at the top.

In [15]:
scale_interaction["discrimination_collapse_detected"] = (
    (scale_interaction["coef"] < 0) & (scale_interaction["p_value"] < 0.05)
)

comparison_table = scale_interaction[[
    "model", "signal_type", "coef", "p_value", "discrimination_collapse_detected"
]].sort_values("p_value")
comparison_table

,model,signal_type,coef,p_value,discrimination_collapse_detected
3,gemma4-e4b,likes_only_noise,0.341236,1.879604e-65,False
4,qwen3-vl-4b,metrics,0.502836,3.358195e-15,False
2,gemma4-e4b,metrics,0.886712,4.155712e-10,False
5,qwen3-vl-4b,likes_only_noise,0.137265,4.158338e-06,False
0,gemma4-12b,metrics,0.070058,1.863091e-04,False
11,ministral-3-14b,likes_only_noise,0.111309,1.802726e-03,False
7,qwen3-vl-8b,likes_only_noise,0.041935,7.761402e-03,False
1,gemma4-12b,likes_only_noise,0.037831,9.444463e-03,False
10,ministral-3-14b,metrics,-0.985766,5.162108e-02,False
6,qwen3-vl-8b,metrics,0.080525,8.314765e-02,False


In [16]:
out_path = REPO_ROOT / "statistical_analysis" / "outputs" / "scale_interaction_test.csv"
comparison_table.to_csv(out_path, index=False)
print(f"Saved to {out_path}")


Saved to /home/dbvis-support/ClaudeProjects/conformity-llms-facebook-posts/statistical_analysis/outputs/scale_interaction_test.csv


## 10. Opposite-corner ratio test (supervisor correction, 2026-08)

**Rewritten — the original sum-based version below (kept in the following markdown cell for the record) was answering the wrong question.** It tested for a genuine correctness effect *on top of* engagement-following. What the supervisor actually wants sized here is the engagement/conformity effect itself: how much does swapping which post has more engagement move the decision, at all.

**The idea.** Take a grid cell `g(X, Y) = P(chose correct | correct_scale=X, incorrect_scale=Y)` for `X < Y` — the "disadvantaged" cell, correct post has *less* engagement — and its mirror `g(Y, X)` — the "advantaged" cell, correct post has *more* engagement. Compute

```
ratio = disadvantaged_pct / advantaged_pct
```

- `ratio == 1` → engagement makes no difference to the decision at all (the model does equally well/poorly whichever post has more engagement) → **no conformity effect**.
- `ratio -> 0` → engagement fully determines the outcome (never picks correct when it's disadvantaged, always when it's advantaged) → **strong conformity effect**.

Both cell percentages get a Haldane-Anscombe continuity correction (`(k+0.5)/(n+1)` instead of `k/n`) before dividing, so a disadvantaged cell that is literally 0% (a real, expected outcome under full conformity) still yields a finite ratio instead of an undefined `0/x`. The single most extreme pair — correct=0 vs. incorrect=1,000,000, the actual top-left/bottom-right corners of the 7x7 grid as plotted — is reported as the headline number per model x signal-type; all 21 off-diagonal mirrored pairs are also computed for the full picture.

In [17]:
def cell_pct(sub: pd.DataFrame, cs: int, ics: int):
    """Aggregate % of images choosing the correct post at grid cell (correct_scale=cs,
    incorrect_scale=ics), plus the Haldane-Anscombe-corrected version used for the ratio
    so a literal 0% or 100% cell never breaks the division."""
    vals = sub[(sub["correct_scale"] == cs) & (sub["incorrect_scale"] == ics)]["chose_correct"]
    n = len(vals)
    if n == 0:
        return None
    k = int(vals.sum())
    return {"n": n, "k": k, "pct": 100 * k / n, "pct_corrected": 100 * (k + 0.5) / (n + 1)}


def opposite_corner_ratio_test(model_dir: str, condition: str):
    """One row per off-diagonal mirrored pair (X<Y): ratio = disadvantaged_pct / advantaged_pct
    (both continuity-corrected). See Section 10 markdown above for why this replaced the sum test."""
    sub = df[(df["model"] == model_dir) & (df["signal_type"] == condition)]
    rows = []
    for cs in SCALES:
        for ics in SCALES:
            if cs >= ics:
                continue  # only X<Y, so "disadvantaged" (cs=X) is unambiguous
            disadv = cell_pct(sub, cs, ics)   # correct=X (small), incorrect=Y (large)
            adv = cell_pct(sub, ics, cs)      # correct=Y (large), incorrect=X (small)
            if disadv is None or adv is None:
                continue
            rows.append({
                "model": model_dir,
                "signal_type": condition,
                "correct_scale_disadv": cs,
                "incorrect_scale_disadv": ics,
                "disadvantaged_pct": disadv["pct"],
                "advantaged_pct": adv["pct"],
                "ratio": disadv["pct_corrected"] / adv["pct_corrected"],
                "n_disadv": disadv["n"],
                "n_adv": adv["n"],
            })
    return rows


ratio_rows = []
for model_dir in MODELS:
    for condition in GRID_CONDITIONS:
        ratio_rows.extend(opposite_corner_ratio_test(model_dir, condition))

corner_ratio_results = pd.DataFrame(ratio_rows)

# Headline number per model x signal-type: the single most extreme pair (0 vs 1,000,000),
# i.e. the actual top-left/bottom-right corners of the 7x7 grid as plotted.
extreme_mask = (corner_ratio_results["correct_scale_disadv"] == SCALES[0]) & \
               (corner_ratio_results["incorrect_scale_disadv"] == SCALES[-1])
corner_ratio_extreme = corner_ratio_results[extreme_mask].sort_values("ratio").reset_index(drop=True)

# Per model x signal-type summary across all 21 mirrored pairs, for the full picture alongside
# the single extreme-corner headline number.
corner_ratio_summary = (
    corner_ratio_results.groupby(["model", "signal_type"])["ratio"]
    .mean()
    .reset_index()
    .rename(columns={"ratio": "mean_ratio_21_pairs"})
    .merge(corner_ratio_extreme[["model", "signal_type", "ratio"]].rename(columns={"ratio": "extreme_ratio_0v1M"}),
           on=["model", "signal_type"])
    .sort_values("extreme_ratio_0v1M")
)
corner_ratio_summary


,model,signal_type,mean_ratio_21_pairs,extreme_ratio_0v1M
3,gemma4-e4b,metrics,0.016907,0.004975
2,gemma4-e4b,likes_only_noise,0.166030,0.004975
5,ministral-3-14b,metrics,0.004995,0.004975
4,ministral-3-14b,likes_only_noise,0.058018,0.005025
9,qwen3-vl-4b,metrics,0.127040,0.005128
10,qwen3-vl-8b,likes_only_noise,0.175467,0.005348
11,qwen3-vl-8b,metrics,0.022660,0.005405
8,qwen3-vl-4b,likes_only_noise,0.055431,0.015385
7,ministral-3-8b,metrics,0.071153,0.017964
6,ministral-3-8b,likes_only_noise,0.659507,0.657143


In [18]:
out_dir = REPO_ROOT / "statistical_analysis" / "outputs"
corner_ratio_results.to_csv(out_dir / "opposite_corner_ratio_test.csv", index=False)
corner_ratio_extreme.to_csv(out_dir / "opposite_corner_ratio_extreme.csv", index=False)
corner_ratio_summary.to_csv(out_dir / "opposite_corner_ratio_summary.csv", index=False)
print(f"Saved to {out_dir}/opposite_corner_ratio_test.csv, "
      f"{out_dir}/opposite_corner_ratio_extreme.csv, "
      f"and {out_dir}/opposite_corner_ratio_summary.csv")


Saved to /home/dbvis-support/ClaudeProjects/conformity-llms-facebook-posts/statistical_analysis/outputs/opposite_corner_ratio_test.csv, /home/dbvis-support/ClaudeProjects/conformity-llms-facebook-posts/statistical_analysis/outputs/opposite_corner_ratio_extreme.csv, and /home/dbvis-support/ClaudeProjects/conformity-llms-facebook-posts/statistical_analysis/outputs/opposite_corner_ratio_summary.csv


### For the record: the original (superseded) sum-based version

Take the same mirrored pair `g(X,Y)`/`g(Y,X)`. If the model's choice were driven *purely* by which post has more engagement — zero independent weight given to which post is actually correct — swapping the numbers should just swap the winner, so `g(X,Y) + g(Y,X)` should equal exactly 100%. A sum meaningfully above 100% was read as evidence of a genuine correctness effect independent of engagement. This is a real and separately interesting question, but it isn't "how big is the conformity effect" — a model could score perfectly on this sum test (100%) while still being 100% engagement-driven, as long as its correctness sensitivity and engagement sensitivity happen to cancel out symmetrically. The ratio test in Section 10 above replaces it as the reported measure of effect size; the code and results below are left as-is for reference, not re-run or relied on going forward.

In [19]:
from scipy.stats import binomtest

def opposite_corner_sign_test(model_dir: str, condition: str):
    sub = df[(df["model"] == model_dir) & (df["signal_type"] == condition)]
    # index by (image_num, correct_scale, incorrect_scale) -> chose_correct
    lookup = {
        (r["image_num"], r["correct_scale"], r["incorrect_scale"]): r["chose_correct"]
        for r in sub.to_dict("records")
    }
    wins = losses = ties = 0
    sums = []
    seen_pairs = set()
    for cs in SCALES:
        for ics in SCALES:
            if cs == ics:
                continue
            pair_key = frozenset({cs, ics})
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)
            for image_num in sub["image_num"].unique():
                a = lookup.get((image_num, cs, ics))
                b = lookup.get((image_num, ics, cs))
                if a is None or b is None:
                    continue
                s = a + b
                sums.append(s)
                if s > 1:
                    wins += 1
                elif s < 1:
                    losses += 1
                else:
                    ties += 1
    n_decisive = wins + losses
    if n_decisive == 0:
        return None
    test = binomtest(wins, n_decisive, 0.5)
    return {
        "model": model_dir,
        "signal_type": condition,
        "mean_sum_pct": 100 * sum(sums) / len(sums),
        "n_wins_excess_correctness": wins,
        "n_losses_deficit": losses,
        "n_ties": ties,
        "p_value": test.pvalue,
    }


corner_rows = []
for model_dir in MODELS:
    for condition in GRID_CONDITIONS:
        r = opposite_corner_sign_test(model_dir, condition)
        if r is not None:
            corner_rows.append(r)

corner_results = pd.DataFrame(corner_rows).sort_values("p_value")
corner_results


,model,signal_type,mean_sum_pct,n_wins_excess_correctness,n_losses_deficit,n_ties,p_value
0,gemma4-12b,metrics,175.047619,1635,59,406,0.000000e+00
1,gemma4-12b,likes_only_noise,167.238095,1488,76,536,0.000000e+00
8,ministral-3-8b,metrics,89.285714,94,319,1687,9.995683e-30
6,qwen3-vl-8b,metrics,96.285714,29,107,1964,1.053144e-11
11,ministral-3-14b,likes_only_noise,96.380952,86,162,1852,1.599344e-06
5,qwen3-vl-4b,likes_only_noise,97.809524,55,101,1944,2.871085e-04
4,qwen3-vl-4b,metrics,102.809524,197,138,1765,1.492781e-03
9,ministral-3-8b,likes_only_noise,94.666667,784,896,420,6.749146e-03
3,gemma4-e4b,likes_only_noise,97.000000,233,296,1571,6.968916e-03
10,ministral-3-14b,metrics,99.619048,0,8,2092,7.812500e-03


**Reading this table:** `mean_sum_pct` well above 100%, paired with a small `p_value`, is evidence of a genuine correctness effect independent of engagement. A `mean_sum_pct` close to 100% (regardless of p-value) is consistent with the "pure engagement, no independent correctness weight" null — i.e. more evidence *against* the model doing any real fact-checking on top of following the crowd.

In [20]:
out_path = REPO_ROOT / "statistical_analysis" / "outputs" / "opposite_corner_test.csv"
corner_results.to_csv(out_path, index=False)
print(f"Saved to {out_path}")


Saved to /home/dbvis-support/ClaudeProjects/conformity-llms-facebook-posts/statistical_analysis/outputs/opposite_corner_test.csv


## 11. Diagonal > 50% test (supervisor suggestion, 2026-07-16)

Extends the per-image diagonal ("competence") scores already used in `THESIS.md` Section 6.1 with an explicit one-sample significance test against chance (50%), rather than only the paired diagonal-vs-above-diagonal comparison already run there. For each model x signal-type combination: average each image's accuracy across its 7 equal-engagement (diagonal) cells, then run an exact sign test on whether each image's diagonal score is above or below 50% (ties at exactly 50% excluded).

In [21]:
def diagonal_above_chance_test(model_dir: str, condition: str):
    sub = df[(df["model"] == model_dir) & (df["signal_type"] == condition) & (df["disparity"] == 0)]
    per_image = sub.groupby("image_num")["chose_correct"].mean()
    wins = int((per_image > 0.5).sum())
    losses = int((per_image < 0.5).sum())
    ties = int((per_image == 0.5).sum())
    n_decisive = wins + losses
    if n_decisive == 0:
        return None
    test = binomtest(wins, n_decisive, 0.5)
    return {
        "model": model_dir,
        "signal_type": condition,
        "mean_diagonal_pct": 100 * per_image.mean(),
        "n_images_above_50": wins,
        "n_images_below_50": losses,
        "n_images_at_50": ties,
        "p_value": test.pvalue,
    }


diagonal_rows = []
for model_dir in MODELS:
    for condition in GRID_CONDITIONS:
        r = diagonal_above_chance_test(model_dir, condition)
        if r is not None:
            diagonal_rows.append(r)

diagonal_results = pd.DataFrame(diagonal_rows).sort_values("p_value")
diagonal_results


,model,signal_type,mean_diagonal_pct,n_images_above_50,n_images_below_50,n_images_at_50,p_value
0,gemma4-12b,metrics,82.714286,89,11,0,2.540853e-16
1,gemma4-12b,likes_only_noise,85.142857,89,11,0,2.540853e-16
4,qwen3-vl-4b,metrics,45.000000,38,62,0,2.097874e-02
2,gemma4-e4b,metrics,52.285714,60,40,0,5.688793e-02
3,gemma4-e4b,likes_only_noise,52.285714,60,40,0,5.688793e-02
5,qwen3-vl-4b,likes_only_noise,44.428571,42,58,0,1.332106e-01
9,ministral-3-8b,likes_only_noise,51.714286,58,42,0,1.332106e-01
8,ministral-3-8b,metrics,51.714286,58,42,0,1.332106e-01
10,ministral-3-14b,metrics,51.714286,58,42,0,1.332106e-01
11,ministral-3-14b,likes_only_noise,51.714286,58,42,0,1.332106e-01


In [22]:
out_path = REPO_ROOT / "statistical_analysis" / "outputs" / "diagonal_above_chance_test.csv"
diagonal_results.to_csv(out_path, index=False)
print(f"Saved to {out_path}")


Saved to /home/dbvis-support/ClaudeProjects/conformity-llms-facebook-posts/statistical_analysis/outputs/diagonal_above_chance_test.csv
